In [1]:
#importing everything needed
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

#Setting up some additional display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [2]:
# calling to dataset
X_train = pd.read_csv("X_train.csv")
y_train = pd.read_csv("y_train.csv")
X_val = pd.read_csv("X_val.csv")
y_val = pd.read_csv("y_val.csv")
X_test = pd.read_csv("X_test.csv")
y_test = pd.read_csv("y_test.csv")

# recombine for EDA purposes
train_data = X_train.copy()
train_data['carbon_emissions'] = y_train

print("Train Shape:", train_data.shape)
display(train_data.head())
display(train_data.info())

Train Shape: (1498, 15)


,Country Name,Year,Access to electricity (% of population),"Agriculture, forestry, and fishing, value added (% of GDP)",Electric power consumption (kWh per capita),Electricity production from coal sources (% of total),Electricity production from hydroelectric sources (% of total),Electricity production from natural gas sources (% of total),Electricity production from nuclear sources (% of total),"Energy imports, net (% of energy use)",Fossil fuel energy consumption (% of total),GDP (constant LCU),"Net trade in goods and services (BoP, current US$)",Urban population,carbon_emissions
0,Algeria,2005,98.6000,6.5593,891.7146,0.0000,1.6364,96.2494,0.0000,-412.4430,99.5923,"5,671,962,184,600.0000","24,200,999,999.9988","21,077,300.0000",101.4002
1,Algeria,2006,98.7000,6.3660,862.8785,0.0000,0.6189,97.2520,0.0000,-373.2893,99.7224,"5,836,449,087,900.0000","31,947,000,000.0014","21,658,728.0000",105.5039
2,Algeria,2007,98.7000,6.3260,893.6976,0.0000,0.6076,97.2578,0.0000,-346.1207,99.7349,"6,017,379,009,700.0000","30,125,606,755.2431","22,280,927.0000",108.9161
3,Algeria,2008,99.3000,5.5498,944.9417,0.0000,0.7034,97.3283,0.0000,-331.6822,99.8040,"6,167,813,484,900.0000","32,476,308,401.8162","22,944,895.0000",113.7088
4,Algeria,2009,98.8000,8.0112,862.4857,0.0000,0.7948,97.6286,0.0000,-274.1430,99.9210,"6,241,827,246,700.0000","-909,806,577.2499","23,604,602.0000",116.0891


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1498 entries, 0 to 1497
Data columns (total 15 columns):
 #   Column                                                          Non-Null Count  Dtype  
---  ------                                                          --------------  -----  
 0   Country Name                                                    1498 non-null   object 
 1   Year                                                            1498 non-null   int64  
 2   Access to electricity (% of population)                         1498 non-null   float64
 3   Agriculture, forestry, and fishing, value added (% of GDP)      1498 non-null   float64
 4   Electric power consumption (kWh per capita)                     1498 non-null   float64
 5   Electricity production from coal sources (% of total)           1498 non-null   float64
 6   Electricity production from hydroelectric sources (% of total)  1498 non-null   float64
 7   Electricity production from natural gas sources (% 

None

In [3]:
# defining target variable and features
target = "carbon_emissions"

X_train_model = X_train.drop(columns=['Country Name', 'Year'])
X_val_model = X_val.drop(columns=['Country Name', 'Year'])
X_test_model = X_test.drop(columns=['Country Name', 'Year'])

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_model)
X_val_scaled = scaler.transform(X_val_model)
X_test_scaled = scaler.transform(X_test_model)

print("Feature columns:")
print(X_train_model.columns.tolist())
print("\nTarget column:")
print(target)

Feature columns:
['Access to electricity (% of population)', 'Agriculture, forestry, and fishing, value added (% of GDP)', 'Electric power consumption (kWh per capita)', 'Electricity production from coal sources (% of total)', 'Electricity production from hydroelectric sources (% of total)', 'Electricity production from natural gas sources (% of total)', 'Electricity production from nuclear sources (% of total)', 'Energy imports, net (% of energy use)', 'Fossil fuel energy consumption (% of total)', 'GDP (constant LCU)', 'Net trade in goods and services (BoP, current US$)', 'Urban population']

Target column:
carbon_emissions


In [4]:
# identifying which columns are numerical and categorical
categorical_features = X_train_model.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X_train_model.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)

Categorical features: []
Numeric features: ['Access to electricity (% of population)', 'Agriculture, forestry, and fishing, value added (% of GDP)', 'Electric power consumption (kWh per capita)', 'Electricity production from coal sources (% of total)', 'Electricity production from hydroelectric sources (% of total)', 'Electricity production from natural gas sources (% of total)', 'Electricity production from nuclear sources (% of total)', 'Energy imports, net (% of energy use)', 'Fossil fuel energy consumption (% of total)', 'GDP (constant LCU)', 'Net trade in goods and services (BoP, current US$)', 'Urban population']


In [5]:
# created a helper function to evalute the models we choose
def evaluate_model(name, model, X_train, X_val, X_test, y_train, y_val, y_test):
    model.fit(X_train, y_train.values.ravel())

    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)

    results = {
        "Model": name,
        "Train R2": r2_score(y_train, y_train_pred),
        "Val R2": r2_score(y_val, y_val_pred),
        "Test R2": r2_score(y_test, y_test_pred),
        "Train RMSE": np.sqrt(mean_squared_error(y_train, y_train_pred)),
        "Val RMSE": np.sqrt(mean_squared_error(y_val, y_val_pred)),
        "Test RMSE": np.sqrt(mean_squared_error(y_test, y_test_pred)),
        "Train MAE": mean_absolute_error(y_train, y_train_pred),
        "Val MAE": mean_absolute_error(y_val, y_val_pred),
        "Test MAE": mean_absolute_error(y_test, y_test_pred)
    }

    return results

In [6]:
# Linear Regression Baseline
linear_model = LinearRegression()

linear_results = evaluate_model(
    "Linear Regression",
    linear_model,
    X_train_model, X_val_model, X_test_model,
    y_train, y_val, y_test
)

linear_results

{'Model': 'Linear Regression',
 'Train R2': 0.932889170937097,
 'Val R2': -1.0673900886492476,
 'Test R2': 0.1977887805990013,
 'Train RMSE': np.float64(312.7514057786004),
 'Val RMSE': np.float64(1586.32676481306),
 'Test RMSE': np.float64(342.72144462295574),
 'Train MAE': 169.72296933157844,
 'Val MAE': 464.73562991704955,
 'Test MAE': 183.07996469496723}

In [7]:
#Based on the results we can see that the R2 for both train and test are not only extremely good but also very similar.
#This may be caused by on hot coding countries, which might be improving the overall model. The closeness between both 
#the training and testing does suggest limited overfitting which is good for the model. 

In [8]:
# Ridge Regression
ridge_alphas = np.logspace(-3, 6, 100)

ridge_model = RidgeCV(alphas=ridge_alphas, cv=5)

ridge_results = evaluate_model(
    "Ridge Regression",
    ridge_model,
    X_train_scaled, X_val_scaled, X_test_scaled,
    y_train, y_val, y_test
)

ridge_results

{'Model': 'Ridge Regression',
 'Train R2': 0.6813529849489826,
 'Val R2': -1.070975024530012,
 'Test R2': 0.42801712491954147,
 'Train RMSE': np.float64(681.4875022697529),
 'Val RMSE': np.float64(1587.7015456504503),
 'Test RMSE': np.float64(289.3932700346939),
 'Train MAE': 227.9899552417487,
 'Val MAE': 479.54321934409705,
 'Test MAE': 182.23981153231603}

In [9]:
# Finding the best alpha for Ridge
best_ridge_alpha = ridge_model.alpha_
print("Best Ridge alpha:", best_ridge_alpha)

Best Ridge alpha: 2310.129700083163


In [10]:
#Ridge Regression created nearly identical results to the baseline model. Since the optimal regularization 
#strength was very small, the Ridge model did not really improve performance, meaning that coefficient shrinkage
#was not really necessary for this dataset.

In [11]:
# Lasso Regression
lasso_alphas = np.logspace(-6, 6, 100)

lasso_model = LassoCV(alphas=lasso_alphas, cv=5, random_state=42, max_iter=20000)

lasso_results = evaluate_model(
    "Lasso Regression",
    lasso_model,
    X_train_scaled, X_val_scaled, X_test_scaled,
    y_train, y_val, y_test
)

lasso_results



{'Model': 'Lasso Regression',
 'Train R2': 0.9358085883352281,
 'Val R2': -1.0475716325699178,
 'Test R2': 0.13782447349544424,
 'Train RMSE': np.float64(305.8732036182165),
 'Val RMSE': np.float64(1578.7050162597595),
 'Test RMSE': np.float64(355.2996334599018),
 'Train MAE': 167.54928115944497,
 'Val MAE': 458.3702977987986,
 'Test MAE': 194.48747969077382}

In [12]:
print("Best Lasso alpha:", lasso_model.alpha_)

feature_coefficients = pd.DataFrame({
    'Feature': X_train_model.columns,
    'Coefficient': lasso_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print(feature_coefficients)

Best Lasso alpha: 1e-06
                                              Feature  Coefficient
11                                   Urban population     981.2868
10  Net trade in goods and services (BoP, current ...     316.7450
9                                  GDP (constant LCU)     -75.9758
1   Agriculture, forestry, and fishing, value adde...     -66.2678
5   Electricity production from natural gas source...     -38.6825
7               Energy imports, net (% of energy use)      21.7485
0             Access to electricity (% of population)     -14.2573
6   Electricity production from nuclear sources (%...     -12.1111
8         Fossil fuel energy consumption (% of total)       7.6232
2         Electric power consumption (kWh per capita)       4.3695
3   Electricity production from coal sources (% of...       3.9663
4   Electricity production from hydroelectric sour...      -2.9291


In [13]:
#Lasso Regression also performed nearly identically to the baseline. While it is useful for feature selection, 
#it still keeping a big number of predictors, especially after one hot encoding country names. This is telling us 
#that many country specific and numeric variables contribute to this predictive performance.

In [14]:
# =========================================
# 11. Compare results
# =========================================
results_df = pd.DataFrame([linear_results, ridge_results, lasso_results])
display(results_df)

,Model,Train R2,Val R2,Test R2,Train RMSE,Val RMSE,Test RMSE,Train MAE,Val MAE,Test MAE
0,Linear Regression,0.9329,-1.0674,0.1978,312.7514,"1,586.3268",342.7214,169.7230,464.7356,183.0800
1,Ridge Regression,0.6814,-1.0710,0.4280,681.4875,"1,587.7015",289.3933,227.9900,479.5432,182.2398
2,Lasso Regression,0.9358,-1.0476,0.1378,305.8732,"1,578.7050",355.2996,167.5493,458.3703,194.4875


In [15]:
#Overall, all three models performed veyr closely to each other, with no major improvement from regularization. 
#This tells us that the dataset already supports a strong linear fit, but some of the predictive 
#power may come from country level identity effects rather than only from the economical development indicators.

In [17]:
# Showing Lasso selected features
coef_df = pd.DataFrame({
    "Feature": X_train_model.columns,
    "Coefficient": lasso_model.coef_
})

selected_features = coef_df[coef_df["Coefficient"] != 0].sort_values(
    by="Coefficient", key=np.abs, ascending=False
)

print("Number of features selected by Lasso:", selected_features.shape[0])
display(selected_features)

Number of features selected by Lasso: 12


,Feature,Coefficient
11,Urban population,981.2868
10,"Net trade in goods and services (BoP, current ...",316.7450
9,GDP (constant LCU),-75.9758
1,"Agriculture, forestry, and fishing, value adde...",-66.2678
5,Electricity production from natural gas source...,-38.6825
7,"Energy imports, net (% of energy use)",21.7485
0,Access to electricity (% of population),-14.2573
6,Electricity production from nuclear sources (%...,-12.1111
8,Fossil fuel energy consumption (% of total),7.6232
2,Electric power consumption (kWh per capita),4.3695
